# Integrity checks

The routines in this notebook perform basic checks on the selected datasets that are easier to be made programmatic:
- checking for NaNs
- checking for aphysical quantities
- dimensions of datasets are what we expect them to be

Eventually it could be useful to operationalize these tests into the production system to make it automated. For now we'll have a section that performs integrity checks on the input datasets and one evaluates output datasets.

In [1]:
%load_ext autoreload
%autoreload 2
import duckdb
import geopandas as gpd
import icechunk
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterix
import seaborn as sns
import xarray as xr

from rasterix.rasterize.rasterio import geometry_clip
import pint
from srm import catalog
from srm.utils import icechunk_store_to_dataset, clean_up_dataset
from srm.qaqc import check_physical_constraints, confirm_coords

In [2]:
catalog

Dataset Catalog (6 datasets)
--------------------------------------------------------------------------------
CESM2-WACCM-Historical-icechunk | icechunk   | s3://carbonplan-srm/input/tensor/CESM2/CESM2-WACCM-Historical/icechunk/icechunk
CESM2-WACCM-G6-1.5K-icechunk | icechunk   | s3://carbonplan-srm/input/tensor/CESM2/CESM2-WACCM-G6-1.5K/icechunk/icechunk
CESM2-WACCM-SSP245-icechunk | icechunk   | s3://carbonplan-srm/input/tensor/CESM2/CESM2-WACCM-SSP245/icechunk/icechunk
MIROC-ES2H-G6-1.5K-icechunk | icechunk   | s3://carbonplan-srm/input/tensor/MIROC-ES2H/MIROC-ES2H-G6-1.5K/MIROC-ES2H-G6-1.5K.icechunk
MIROC-ES2H-baseline-icechunk | icechunk   | s3://carbonplan-srm/input/tensor/MIROC-ES2H/MIROC-ES2H-baseline/MIROC-ES2H-baseline.icechunk
ERA5                 | icechunk   | s3://carbonplan-srm/input/tensor/ERA5/era5_rechunked_resampled.icechunk

# Get input data

Once the input datasets are in their final place for use and are all done with any pre-processing (e.g. rechunking), they won't change. So, we only have to run the integrity routines once and they can live in their own separate notebook.

### Output from three climate model simulations

These simulations were conducted in the Community Earth System Model version 2 -- Whole Atmosphere Community Climate Model (CESM2-WACCM)
* Historical (1978 - 2014)
* SSP245 (2015 - 2070)
* G6-1.5K (2035 - 2085)

In [3]:
gcm = "CESM2-WACCM"
scenario = "SSP245" # "G6-1.5K" , "Historical"
variables = ['tas', 'tasmin', 'tasmax', 'pr', 'rsds']
dataset_name = f"{gcm}-{scenario}-icechunk"

In [4]:
ds = icechunk_store_to_dataset(dataset_name, catalog)

In [5]:
ds = clean_up_dataset(ds, gcm)[variables]

### Observations: ERA5

In [6]:
era5 = icechunk_store_to_dataset("ERA5", catalog)

In [7]:
era5 = clean_up_dataset(era5, "ERA5")[variables]

### Integrity checks

In [8]:
# test running on a 2x2 box for expediency since the dataset is chunked along full time dimension
# and so would take a long time to check fully
out = ds.sel(latitude=slice(46, 48), 
                  longitude=slice(-124, -122.0)).sel(ensemble_member='006').load()

In [9]:
check_physical_constraints(out)

Number of negative precipitation values: 0
Number of outlandishly high precipitation values: 0
Number of outlandishly high tas values: 0
Number of outlandishly high tasmax values: 0
Number of outlandishly high tasmin values: 0
Number of outlandishly low tas values: 0
Number of outlandishly low tasmax values: 0
Number of outlandishly low tasmin values: 0
Number of times tasmin exceeds tas: 0
Number of times tas exceeds tasmax: 0
Number of times tasmin exceeds tasmax: 0


In [10]:
check_physical_constraints(era5.sel(latitude=slice(47, 48), 
                  longitude=slice(-122, -121)))

Number of negative precipitation values: 20693
Number of outlandishly high precipitation values: 0
Number of outlandishly high tas values: 0
Number of outlandishly high tasmax values: 0
Number of outlandishly high tasmin values: 0
Number of outlandishly low tas values: 0
Number of outlandishly low tasmax values: 0
Number of outlandishly low tasmin values: 0
Number of times tasmin exceeds tas: 89
Number of times tas exceeds tasmax: 5440
Number of times tasmin exceeds tasmax: 0


In [11]:
confirm_coords(era5)

'Latitude and longitude match expectation'